# NetOps-7B: QLoRA Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwccie/netops-7b/blob/main/notebooks/02_training.ipynb)

Fine-tune Qwen 2.5 7B Instruct with QLoRA for network operations.

**Requirements:** GPU runtime (T4 minimum, A100 recommended)

## Training Configuration

| Parameter | Value |
|-----------|-------|
| Base Model | Qwen/Qwen2.5-7B-Instruct |
| Quantization | 4-bit NF4 |
| LoRA Rank | 16 |
| LoRA Alpha | 32 |
| Target Modules | All linear layers |
| Epochs | 3 |
| Learning Rate | 2e-4 |
| Batch Size | 2 (effective 8) |
| Optimizer | AdamW 8-bit |

In [ ]:
# Install dependencies
!pip install -q git+https://github.com/cwccie/netops-7b.git
!pip install -q unsloth

In [ ]:
from netops_7b.train.qlora import QLoRAConfig, QLoRATrainer

# Configure training
config = QLoRAConfig(
    dataset_path="data/train.jsonl",
    output_dir="outputs/netops-7b-qlora",
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
)

print(f"Effective batch size: {config.effective_batch_size}")

In [ ]:
# Initialize trainer
trainer = QLoRATrainer(config)
trainer.setup()

# Print trainable parameters
params = trainer.print_trainable_params()
print(f"Trainable: {params['trainable_params']:,} / {params['total_params']:,} ({params['trainable_percent']}%)")

In [ ]:
# Run training
metrics = trainer.train()
print(f"Training complete! Loss: {metrics.get('train_loss', 'N/A')}")

# Save model
save_path = trainer.save()
print(f"Model saved to: {save_path}")

In [ ]:
# Export to GGUF
from netops_7b.train.export import GGUFExporter

exporter = GGUFExporter(
    model_path=str(save_path),
    output_dir="outputs/gguf",
)
exported = exporter.export_unsloth(["q4_k_m", "q5_k_m", "q8_0"])
for quant, path in exported.items():
    print(f"{quant}: {path}")